# Annotation Quality Audit Program

This program audits the annotation quality of dogwhistle terms in the benchmark dataset.
It decomposes annotations into cases A (present + hateful), B (present + non-hateful),
and C (absent from union), and calculates labeling accuracy metrics.

In [1]:
# Imports
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional

import pandas as pd
import re

In [2]:
@dataclass
class AnnotationQualityAuditConfig:
    # `workdir` is the anchor used to resolve all relative paths.
    workdir: Path

    # Local sources expected to exist in the repository.
    data_local_path: Path = Path('../outputs/preprocessing/dedup_primary.tsv')
    glossary_local_path: Path = Path('../data/glossary.tsv')

    # Directory where audit exports are written.
    output_dir: Path = Path('../outputs/annotation_audits')

In [3]:
# Configuration
WORKDIR = Path.cwd()
cfg = AnnotationQualityAuditConfig(workdir=WORKDIR)

# Convert all configured relative paths into absolute paths once up front.
cfg.output_dir = cfg.workdir / cfg.output_dir
cfg.data_path = cfg.workdir / cfg.data_local_path
cfg.glossary_path = cfg.workdir / cfg.glossary_local_path

# Create output/cache directories early so later cells can assume they exist.
cfg.output_dir.mkdir(parents=True, exist_ok=True)

cfg

AnnotationQualityAuditConfig(workdir=WindowsPath('c:/Users/ryanf/Documents/GitHub/benchmarking_dogwhistles'), data_local_path=WindowsPath('../outputs/preprocessing/dedup_primary.tsv'), glossary_local_path=WindowsPath('../data/glossary.tsv'), output_dir=WindowsPath('c:/Users/ryanf/Documents/GitHub/benchmarking_dogwhistles/outputs/annotation_audits'))

In [4]:
# Load data
glossary = pd.read_csv(cfg.glossary_path, sep='\t')
data = pd.read_csv(cfg.data_path, sep='\t')

# Ensure everything is lowercase for matching
glossary['surface_form'] = glossary['surface_form'].str.strip().str.lower()
glossary['target'] = glossary['target'].str.strip().str.lower()

# Compile regex: Sorting by length (descending) prevents partial matches 
# (e.g., matching 'dog' inside 'dogwhistle')
all_forms = sorted(glossary['surface_form'].unique(), key=len, reverse=True)
pattern = re.compile(r'\b(' + '|'.join(map(re.escape, all_forms)) + r')\b', flags=re.IGNORECASE)

In [5]:
# 1. Extract matches
data['found_forms'] = data['text'].apply(lambda x: pattern.findall(x.lower()) if pd.notna(x) else [])

# 2. Explode and Join
# Each row in 'matches_df' represents one instance of a found dogwhistle
matches_df = data.explode('found_forms').dropna(subset=['found_forms'])

# 3. Merge with glossary to get the taxonomy_level and target for each match
# This handles cases where one surface_form might belong to multiple categories
audit_df = matches_df.merge(
    glossary, 
    left_on='found_forms', 
    right_on='surface_form', 
    how='inner'
)

In [6]:
# Build annotation quality metrics
# For each level/target, decompose into case A (present + hateful), 
# B (present + non-hateful), and C (absent from union)

annotation_quality_data = []

for (level, target), group in audit_df.groupby(['taxonomy_level', 'target']):
    # Case A: Posts with forms from this level/target labeled as hateful
    case_a = len(group[group['binary_hate'] == 1])
    
    # Case B: Posts with forms from this level/target labeled as non-hateful
    case_b = len(group[group['binary_hate'] == 0])
    
    # Case C: Posts that target this group but DON'T contain any forms 
    # from this level/target combination
    # Start with all posts targeting this group
    target_group_posts = data[data['targets'].str.contains(target, case=False, na=False)]
    
    # Find posts that contain any form from this glossary subset
    glossary_subset = glossary[(glossary['taxonomy_level'] == level) & 
                                (glossary['target'] == target)]
    forms_for_level_target = set(glossary_subset['surface_form'].unique())
    
    # Posts in target group that have forms from this level/target
    posts_with_forms = group['text_dedup_key'].unique()
    
    # Case C: Target group posts without forms from this level/target=
    case_c = len(target_group_posts[~target_group_posts['text_dedup_key'].isin(posts_with_forms)])
    
    # Calculate metrics
    matches_with_label = case_a + case_b
    correct_labeling_rate = case_a / matches_with_label if matches_with_label > 0 else 0
    annotator_failure_ratio = case_b / matches_with_label if matches_with_label > 0 else 0
    
    annotation_quality_data.append({
        'taxonomy_level': level,
        'target': target,
        'case_a_present_hateful': case_a,
        'case_b_present_nonhateful': case_b,
        'case_c_absent': case_c,
        'correct_labeling_rate': correct_labeling_rate,
        'annotator_failure_ratio': annotator_failure_ratio,
        'total_matches': matches_with_label,
        'total_target_group_posts': len(target_group_posts)
    })

annotation_quality_report = pd.DataFrame(annotation_quality_data)
annotation_quality_report

,taxonomy_level,target,case_a_present_hateful,case_b_present_nonhateful,case_c_absent,correct_labeling_rate,annotator_failure_ratio,total_matches,total_target_group_posts
0,2,african,0,2,4468,0.000000,1.000000,2,4468
1,3,african,1,34,4453,0.028571,0.971429,35,4468
2,3,hispanic,0,1,711,0.000000,1.000000,1,711
3,4,transgender women,1,2,1476,0.333333,0.666667,3,1476


In [7]:
# Analyze case breakdown: B/(B+C) indicates whether gap is annotation vs. collection
case_breakdown = annotation_quality_report.copy()
case_breakdown['case_b_c_ratio'] = (
    case_breakdown['case_b_present_nonhateful'] / 
    (case_breakdown['case_b_present_nonhateful'] + case_breakdown['case_c_absent'])
)
case_breakdown['collection_gap_prop'] = (
    case_breakdown['case_c_absent'] / 
    (case_breakdown['case_b_present_nonhateful'] + case_breakdown['case_c_absent'])
)

print("=" * 100)
print("ANNOTATION QUALITY AUDIT: CASE BREAKDOWN")
print("=" * 100)
print("\nCase B/C Ratio: Case B / (Case B + Case C)")
print("  - Closer to 1.0: primarily annotation failure (forms are there, missed by annotators)")
print("  - Closer to 0.0: primarily collection failure (forms not in benchmark at all)")
print("\n")
print(case_breakdown[['taxonomy_level', 'target', 'case_a_present_hateful', 
                       'case_b_present_nonhateful', 'case_c_absent', 
                       'case_b_c_ratio', 'collection_gap_prop']].to_string(index=False))

print("\n" + "=" * 100)
print("INTERPRETATION SUMMARY")
print("=" * 100)
for idx, row in case_breakdown.iterrows():
    level = row['taxonomy_level']
    target = row['target']
    ratio = row['case_b_c_ratio']
    b = row['case_b_present_nonhateful']
    c = row['case_c_absent']
    
    if b + c > 0:
        if ratio > 0.7:
            problem = "ANNOTATION FAILURE (primary issue is mislabeling)"
        elif ratio < 0.3:
            problem = "COLLECTION FAILURE (primary issue is missing forms)"
        else:
            problem = "MIXED (both annotation and collection issues)"
    else:
        problem = "INSUFFICIENT DATA"
    
    print(f"\nLevel {level}, {target.upper()}: {problem}")
    print(f"  Case A (correct): {row['case_a_present_hateful']} | " +
          f"Case B (missed): {b} | Case C (absent): {c}")

ANNOTATION QUALITY AUDIT: CASE BREAKDOWN

Case B/C Ratio: Case B / (Case B + Case C)
  - Closer to 1.0: primarily annotation failure (forms are there, missed by annotators)
  - Closer to 0.0: primarily collection failure (forms not in benchmark at all)


 taxonomy_level            target  case_a_present_hateful  case_b_present_nonhateful  case_c_absent  case_b_c_ratio  collection_gap_prop
              2           african                       0                          2           4468        0.000447             0.999553
              3           african                       1                         34           4453        0.007577             0.992423
              3          hispanic                       0                          1            711        0.001404             0.998596
              4 transgender women                       1                          2           1476        0.001353             0.998647

INTERPRETATION SUMMARY

Level 2, AFRICAN: COLLECTION FAILUR

In [9]:
# Detailed breakdown by surface form: which forms are mislabeled (case B)
form_labeling_detail = []

for (level, target), group in audit_df.groupby(['taxonomy_level', 'target']):
    for form in group['found_forms'].unique():
        form_data = group[group['found_forms'] == form]
        
        hateful_count = len(form_data[form_data['binary_hate'] == 1])
        nonhateful_count = len(form_data[form_data['binary_hate'] == 0])
        
        if hateful_count + nonhateful_count > 0:
            labeling_accuracy = hateful_count / (hateful_count + nonhateful_count)
        else:
            labeling_accuracy = 0
        
        form_labeling_detail.append({
            'taxonomy_level': level,
            'target': target,
            'surface_form': form,
            'case_a_count': hateful_count,
            'case_b_count': nonhateful_count,
            'labeling_accuracy': labeling_accuracy,
            'type': form_data['type'].iloc[0] if len(form_data) > 0 else 'unknown'
        })

form_labeling_report = pd.DataFrame(form_labeling_detail).sort_values(
    by=['taxonomy_level', 'target', 'case_b_count'], ascending=[True, True, False])

print("\n" + "=" * 100)
print("SURFACE FORM LABELING DETAILS (Case A vs Case B)")
print("=" * 100)
print("\nForms sorted by mislabeling frequency (Case B count):")
print(form_labeling_report.to_string(index=False))


SURFACE FORM LABELING DETAILS (Case A vs Case B)

Forms sorted by mislabeling frequency (Case B count):
 taxonomy_level            target               surface_form  case_a_count  case_b_count  labeling_accuracy                                type
              2           african            lack of fathers             0             1           0.000000 stereotype-based target group label
              2           african            absentee father             0             1           0.000000 stereotype-based target group label
              3           african         affirmative action             1            34           0.028571                    concept (policy)
              3          hispanic end birthright citizenship             0             1           0.000000                    concept (policy)
              4 transgender women               actual women             1             1           0.500000   persona signal (self-referential)
              4 transgender wom

In [10]:
# Export annotation quality audit results
annotation_quality_path = cfg.output_dir / 'annotation_quality.tsv'
case_breakdown_path = cfg.output_dir / 'case_breakdown.tsv'
form_labeling_path = cfg.output_dir / 'form_labeling_detail.tsv'

annotation_quality_report.to_csv(annotation_quality_path, sep='\t', index=False)
case_breakdown.to_csv(case_breakdown_path, sep='\t', index=False)
form_labeling_report.to_csv(form_labeling_path, sep='\t', index=False)

print("\nAnnotation quality audit exported:")
print(f"  - {annotation_quality_path}")
print(f"  - {case_breakdown_path}")
print(f"  - {form_labeling_path}")


Annotation quality audit exported:
  - c:\Users\ryanf\Documents\GitHub\benchmarking_dogwhistles\outputs\annotation_audits\annotation_quality.tsv
  - c:\Users\ryanf\Documents\GitHub\benchmarking_dogwhistles\outputs\annotation_audits\case_breakdown.tsv
  - c:\Users\ryanf\Documents\GitHub\benchmarking_dogwhistles\outputs\annotation_audits\form_labeling_detail.tsv
